In [1]:
!pip uninstall -y torchao
!pip install -q peft accelerate evaluate

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00


In [2]:
# Fraud/Phishing Text Classification — LoRA Fine-Tuning
# Run this in Google Colab (free T4 GPU: Runtime > Change runtime type > T4 GPU)
# 100% free, open-source stack: Qwen2.5-0.5B + HF PEFT (LoRA) + public HF dataset

# ============================================================
# STEP 1: Install dependencies (Colab already has torch)
# ============================================================
# !pip install -q transformers peft datasets accelerate bitsandbytes evaluate scikit-learn

import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from peft import LoraConfig, get_peft_model, TaskType
import numpy as np
import evaluate

# ============================================================
# STEP 2: Load dataset
# ============================================================
# Public HF dataset — phishing vs legitimate text/email classification
# If this exact dataset name has issues, swap for any public phishing/spam
# classification dataset on huggingface.co/datasets (search "phishing" or "spam")
dataset = load_dataset("zefang-liu/phishing-email-dataset")

# Inspect columns once to confirm text/label field names before running fully
print(dataset)
print(dataset["train"][0])

# NOTE: Adjust these two lines if the actual column names differ
TEXT_COL = "Email Text"
LABEL_COL_RAW = "Email Type"
LABEL_COL = "label"

dataset = dataset.filter(lambda x: x[TEXT_COL] is not None)

label_names = sorted(set(dataset["train"][LABEL_COL_RAW]))
label2id = {name: i for i, name in enumerate(label_names)}
print("Label mapping:", label2id)

def encode_labels(example):
    example[LABEL_COL] = label2id[example[LABEL_COL_RAW]]
    return example

dataset = dataset.map(encode_labels)
# ============================================================
# STEP 3: Tokenizer + preprocessing
# ============================================================
MODEL_NAME = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def preprocess(examples):
    return tokenizer(
        examples[TEXT_COL],
        truncation=True,
        max_length=256,
        padding=False,
    )

tokenized = dataset.map(preprocess, batched=True)

cols_to_remove = [c for c in tokenized["train"].column_names
                   if c not in ("input_ids", "attention_mask", LABEL_COL)]
tokenized = tokenized.remove_columns(cols_to_remove)
tokenized = tokenized.rename_column(LABEL_COL, "labels")

split = tokenized["train"].train_test_split(test_size=0.15, seed=42)
tokenized = split

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# ============================================================
# STEP 4: Load base model + apply LoRA
# ============================================================
num_labels = len(set(tokenized["train"]["labels"]))

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
)
model.config.pad_token_id = tokenizer.pad_token_id

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],  # standard for Qwen2/Llama-style models
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # sanity check — should be a small % of total params

# ============================================================
# STEP 5: Evaluation metric
# ============================================================
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)
    f1 = f1_metric.compute(predictions=preds, references=labels, average="weighted")
    return {"accuracy": acc["accuracy"], "f1": f1["f1"]}

# ============================================================
# STEP 6: Train
# ============================================================
training_args = TrainingArguments(
    output_dir="./lora-fraud-classifier",
    learning_rate=2e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=20,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# BEFORE fine-tuning: evaluate the base (near-random, since head is untrained) — optional baseline
# baseline_metrics = trainer.evaluate()
# print("Baseline (pre-training):", baseline_metrics)

trainer.train()

# AFTER fine-tuning
final_metrics = trainer.evaluate()
print("Final metrics after LoRA fine-tuning:", final_metrics)

# ============================================================
# STEP 7: Save adapter (small file — just the LoRA weights)
# ============================================================
model.save_pretrained("./lora-fraud-classifier-adapter")
tokenizer.save_pretrained("./lora-fraud-classifier-adapter")

print("Done. Push the ./lora-fraud-classifier-adapter folder + this script to a GitHub repo.")
print("Use final_metrics (accuracy/f1) as your real, honest resume numbers.")

README.md:   0%|          | 0.00/616 [00:00<?, ?B/s]

Phishing_Email.csv:   0%|          | 0.00/52.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/18650 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'Email Text', 'Email Type'],
        num_rows: 18650
    })
})
{'Unnamed: 0': 0, 'Email Text': 're : 6 . 1100 , disc : uniformitarianism , re : 1086 ; sex / lang dick hudson \'s observations on us use of \'s on \' but not \'d aughter \' as a vocative are very thought-provoking , but i am not sure that it is fair to attribute this to " sons " being " treated like senior relatives " . for one thing , we do n\'t normally use \' brother \' in this way any more than we do \'d aughter \' , and it is hard to imagine a natural class comprising senior relatives and \'s on \' but excluding \' brother \' . for another , there seem to me to be differences here . if i am not imagining a distinction that is not there , it seems to me that the senior relative terms are used in a wider variety of contexts , e . g . , calling out from a distance to get someone \'s attention , and hence at the beginning of an utterance , whereas \'s on 

Filter:   0%|          | 0/18650 [00:00<?, ? examples/s]

...

trainable params: 542,464 || all params: 494,577,024 || trainable%: 0.1097


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.066140,0.170127,0.967811,0.967599
2,0.150342,0.165165,0.980329,0.980368


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Final metrics after LoRA fine-tuning: {'eval_loss': 0.1651647388935089, 'eval_accuracy': 0.9803290414878397, 'eval_f1': 0.9803682458745576, 'eval_runtime': 143.039, 'eval_samples_per_second': 19.547, 'eval_steps_per_second': 1.223, 'epoch': 2.0}
Done. Push the ./lora-fraud-classifier-adapter folder + this script to a GitHub repo.
Use final_metrics (accuracy/f1) as your real, honest resume numbers.


In [3]:
import shutil
shutil.make_archive("lora_fraud_classifier_adapter", "zip", "./lora-fraud-classifier-adapter")
print("Zipped successfully.")

Zipped successfully.
